# Lab 2: From Raw Data to ML-Ready Data
## Data Cleaning & Exploratory Data Analysis — Telco Customer Churn

**Name:** Sineha Kumari

**Roll Number:** 023-24-0024

**GitHub Repository:** https://github.com/SinehaKumari/Lab-2-Telco-Customer-Churn

**Duration:** 3 Hours (independent, hands-on)

> This notebook is a **template**, not a tutorial. You already know Python, NumPy, Pandas, and Matplotlib from previous labs. Each section below states the objective and the questions you must answer.
>
> Fill in the empty code cells and the *Answer:* / *Observation:* placeholders directly in this notebook. Do not delete the Markdown headings — they are used for grading.

---

## Setup

Import the libraries you'll need and load the dataset. Use the Telco Customer Churn CSV provided for this lab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('Telco_customer_churn.csv')
print('✅ Dataset loaded successfully!')
print(f'Shape: {df.shape}')

---\n## Part 1 — The Business Problem\n\nA telecommunications company is losing customers to competitors — this is called **customer churn**.

**Answer 1:** A telecom company is experiencing customer churn — customers are leaving for competitor services. The business needs to identify which customers are at risk of churning so proactive retention strategies can be implemented. The target variable is likely the 'Churn' column (Yes/No).

In [ ]:
# Task 4 — dataset shape, column names, data types
print('='*70)
print('DATASET STRUCTURE OVERVIEW')
print('='*70)
print(f'\nDataset Shape: {df.shape}')
print(f'Total Customers: {df.shape[0]}')
print(f'Total Features: {df.shape[1]}')

print('\n' + '='*70)
print('DATA TYPES:')
print('='*70)
print(df.dtypes)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print('\nNumerical Features:', numerical_cols)
print('\nCategorical Features:', categorical_cols)

In [ ]:
# Task 8 — duplicate rows vs duplicate IDs
print('='*70)
print('DUPLICATE VALUES ANALYSIS')
print('='*70)

duplicated_rows = df.duplicated().sum()
print(f'\nCompletely Duplicated Rows: {duplicated_rows}')
if duplicated_rows > 0:
    print('Duplicate rows found:')
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
else:
    print('✅ No completely duplicate rows')

duplicate_ids = df['customerID'].duplicated().sum()
print(f'\nDuplicate Customer IDs: {duplicate_ids}')
if duplicate_ids > 0:
    print('Customers appearing multiple times:')
    dup_customers = df[df['customerID'].duplicated(keep=False)].sort_values('customerID')
    print(dup_customers)
else:
    print('✅ All customer IDs are unique')

In [ ]:
# Task 9 — investigate suspicious values
print('='*70)
print('SUSPICIOUS VALUES INVESTIGATION')
print('='*70)

print('\nTenure Analysis:')
print('-' * 70)
print(f'Min: {df["tenure"].min()}')
print(f'Max: {df["tenure"].max()}')
print(f'Mean: {df["tenure"].mean():.2f}')
print(f'Negative values: {(df["tenure"] < 0).sum()}')
print(f'Zero values: {(df["tenure"] == 0).sum()}')

print('\nTotalCharges Analysis:')
print('-' * 70)
print(f'Data Type: {df["TotalCharges"].dtype}')
try:
    total_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
    non_numeric_count = total_numeric.isnull().sum() - df['TotalCharges'].isnull().sum()
    print(f'Non-numeric values: {non_numeric_count}')
    if non_numeric_count > 0:
        print('\n⚠️ SUSPICIOUS: TotalCharges contains non-numeric values!')
        mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isnull() & (df['TotalCharges'].notna())
        if mask.sum() > 0:
            print('Examples of problematic values:')
            print(df[mask][['customerID', 'TotalCharges', 'tenure', 'Churn']].head())
except Exception as e:
    print(f'Could not convert to numeric: {e}')

## Part 4 — Data Cleaning

| Problem | Decision | Method Used | Reason |
|---|---|---|---|
| TotalCharges stored as object (text) | Convert to numeric | `pd.to_numeric(..., errors='coerce')` | ML models require numeric types |
| Missing values in TotalCharges | Delete rows | `df.dropna(subset=['TotalCharges'])` | Only 0.16% of data |
| Negative tenure values | Delete rows | `df[df['tenure'] >= 0]` | Negative tenure is impossible |

In [ ]:
# Task 11 — apply cleaning decisions
print('='*70)
print('APPLYING CLEANING DECISIONS')
print('='*70)

clean_df = df.copy()
print(f'\nStarting shape: {clean_df.shape}')

# Step 1: Convert TotalCharges to numeric
print('\nStep 1: Convert TotalCharges to numeric')
clean_df['TotalCharges'] = pd.to_numeric(clean_df['TotalCharges'], errors='coerce')
print(f'✅ Converted. Missing values after conversion: {clean_df["TotalCharges"].isnull().sum()}')

# Step 2: Drop rows with missing TotalCharges
print('\nStep 2: Drop rows with missing TotalCharges')
clean_df = clean_df.dropna(subset=['TotalCharges'])
print(f'✅ Dropped. New shape: {clean_df.shape}')

# Step 3: Drop rows with negative tenure
print('\nStep 3: Remove negative tenure values')
clean_df = clean_df[clean_df['tenure'] >= 0]
print(f'✅ Cleaned. Final shape: {clean_df.shape}')

# Step 4: Drop customerID column
print('\nStep 4: Drop identifier columns')
clean_df = clean_df.drop('customerID', axis=1)
print(f'✅ Dropped customerID. Columns now: {len(clean_df.columns)}')

print('\n' + '='*70)
print('FINAL VERIFICATION')
print('='*70)
print(f'\nCleaned Dataset Shape: {clean_df.shape}')
print(f'Total Missing Values: {clean_df.isnull().sum().sum()}')
print(f'Duplicates: {clean_df.duplicated().sum()}')
print('\nData Types:')
print(clean_df.dtypes)
clean_df.info()

In [ ]:
# Task 12 — Churn distribution
print('='*70)
print('CHURN DISTRIBUTION ANALYSIS')
print('='*70)

churn_counts = clean_df['Churn'].value_counts()
churn_pct = clean_df['Churn'].value_counts(normalize=True) * 100

print('\nChurn Counts:')
for category, count in churn_counts.items():
    pct = churn_pct[category]
    print(f'{category}: {count:,} customers ({pct:.2f}%)')

print(f'\nTotal Customers: {len(clean_df):,}')
print(f'Overall Churn Rate: {churn_pct.get("Yes", 0):.2f}%')